# Stacking y cascading: combinar modelos de naturaleza distinta

> Cómo un metamodelo aprende a combinar las predicciones de modelos de base diferente (regresión logística, vecinos, bosques, *boosting*…); por qué las predicciones que se le dan deben ser fuera de muestra, cuándo aporta algo frente a elegir el mejor modelo y qué añade el *cascading* (pasarle también los datos originales).
>
> **Requisitos previos:** [Flujo de ML y evaluación](01-flujo-ml-y-evaluacion.ipynb) · [Bagging y Random Forest](10-bagging-y-random-forest.ipynb) · [Boosting](11-boosting.ipynb) · **Relacionado:** [k vecinos más cercanos](07-knn.ipynb) · [interpretabilidad](14-interpretabilidad.ipynb)

## 1. Idea clave

El [bagging](10-bagging-y-random-forest.ipynb) y el [boosting](11-boosting.ipynb) combinan muchas copias del **mismo tipo** de modelo (árboles). El **stacking** (*apilamiento*) combina modelos **de naturaleza distinta** (uno lineal, uno basado en vecinos, un bosque…) con un último modelo, el **metamodelo**, que no vota sino que **aprende** cuánto fiarse de cada uno a partir de sus predicciones. El **cascading** (*cascada*) es la misma idea, pero el metamodelo recibe además los datos originales, de modo que puede decidir en qué zonas del espacio confiar en cada modelo.

Merece la pena cuando tienes varios modelos razonablemente buenos que **se equivocan en sitios distintos**; no merece la pena si uno de ellos ya domina o si los modelos son casi copias. Es la técnica más cara y la más fácil de hacer mal (fuga de datos), y su mejora suele ser pequeña: este notebook intenta medirla de forma honesta.

## 2. Conceptos importantes

### 2.1. Intuición: un árbitro que aprende en quién confiar

Imagina cuatro peritos que valoran el mismo caso. Una votación simple da a todos el mismo peso. Un buen árbitro, en cambio, ha visto durante meses quién acierta y cuándo: "si el perfil A dice que sí y el B dice que no, suele tener razón B". El metamodelo es ese árbitro: un modelo de aprendizaje automático cuyas **variables de entrada son las predicciones de los modelos base** y cuya salida es la clase final.

| | Bagging / Random Forest | Boosting | Stacking |
|---|---|---|---|
| Modelos base | Copias del mismo algoritmo, entrenadas con muestras distintas | Árboles débiles encadenados | Algoritmos **distintos**, cada uno completo |
| Cómo se combinan | Promedio o voto, pesos iguales | Suma ponderada fija por el algoritmo | Un **metamodelo** entrenado aprende los pesos |
| Qué reduce | Varianza | Sesgo | Un poco de ambos, si los modelos son complementarios |

### 2.2. El algoritmo

Sean $B$ modelos base $f_1,\dots,f_B$ y un conjunto de entrenamiento $\{(x_i, y_i)\}_{i=1}^{n}$.

1. Se divide el entrenamiento en $K$ particiones (*folds*). Para cada modelo $f_b$ y cada ejemplo $i$, se calcula la predicción $\hat{p}_{b}^{(-k(i))}(x_i)$ con una versión de $f_b$ entrenada **sin** la partición $k(i)$ a la que pertenece el ejemplo (predicción *fuera de muestra*, `cross_val_predict`).
2. Con ellas se construye la matriz de metavariables $z_i = \big(\hat{p}_1^{(-k(i))}(x_i), \dots, \hat{p}_B^{(-k(i))}(x_i)\big)$ y se entrena el metamodelo $g$ sobre $\{(z_i, y_i)\}$.
3. Cada $f_b$ se vuelve a entrenar con **todo** el conjunto de entrenamiento.
4. Para un ejemplo nuevo: $\hat{y}(x) = g\big(f_1(x), \dots, f_B(x)\big)$.

Donde $\hat{p}_b$ es la probabilidad de la clase positiva que da el modelo $b$ (en multiclase, un vector de probabilidades por modelo).

**Por qué fuera de muestra.** Un modelo flexible acierta casi siempre sobre los datos con los que se entrenó (un bosque profundo o 1-NN dan probabilidad 1 a la clase correcta). Si el metamodelo aprendiera con esas predicciones, concluiría que ese modelo es infalible y le daría casi todo el peso; con datos nuevos, el modelo falla y el metamodelo falla con él. Es una **fuga de datos** (*data leakage*) y se ve en 3.2.

### 2.3. Qué le damos al metamodelo

- **Probabilidades** (o puntuaciones de decisión) mejor que clases: llevan la confianza de cada modelo.
- Un metamodelo **simple** (regresión logística, regresión lineal en regresión): tiene pocas entradas muy correlacionadas y se sobreajusta con facilidad. Los pesos además se pueden leer.

### 2.4. Cascading: pasar también los datos originales

En *stacking* el metamodelo ve solo $z_i$. En *cascading* ve $[z_i, x_i]$: las decisiones parciales **y los datos de los que salieron**, y puede aprender reglas del tipo "si $x_j$ es grande, fíate del bosque; si no, del modelo lineal". En scikit-learn se hace con `passthrough=True`. Es una aproximación: en la formulación general del cascading también pueden pasarse otros datos generados durante la decisión (por ejemplo, la profundidad de la hoja a la que llegó un árbol o la distancia a los centroides de un *clustering*), cosa que `passthrough` no hace. Por eso el metamodelo debe ser capaz de usar las variables originales (un árbol o un bosque; con uno lineal, `passthrough` apenas cambia nada).

### 2.5. Cuándo funciona

El error de un conjunto de modelos baja cuando los modelos son **buenos** y sus errores **poco correlacionados**. Si todos son casi iguales, el metamodelo no tiene nada nuevo que aprender; si uno es mucho mejor que los demás, el metamodelo acabará dándole casi todo el peso y el resultado será, como mucho, el de ese modelo. Lo comprobaremos con las correlaciones entre las predicciones de los modelos (3.4).

## 3. Código

Usamos un conjunto **sintético** de 2000 filas y 10 variables construido para que los modelos sean complementarios: la clase depende de forma **lineal** de `x0` y `x1`, y de una **interacción** entre `x2` y `x3` (que las dos tengan el mismo signo), más ruido logístico. Un modelo lineal captura la primera parte y no la segunda; los árboles, al revés. Además, al final, **breast cancer** (569 tumores, `load_breast_cancer`, derivado del repositorio UCI), el conjunto limpio que ya usamos en [árboles](08-arboles-decision.ipynb), [bagging](10-bagging-y-random-forest.ipynb) y [boosting](11-boosting.ipynb).

Como los cuatro modelos base trabajan con distintas escalas, los que lo necesitan (regresión logística, kNN) llevan un `StandardScaler` dentro de un `Pipeline` (ver [transformación de datos](../03-preparacion-datos/03-transformacion-datos.ipynb)); los árboles no lo necesitan. La métrica es el $F_1$ macro.

In [1]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### 3.1. Datos y modelos base

Reservamos un 40 % para test y comparamos los cuatro modelos base con validación cruzada de 5 particiones sobre el entrenamiento.

In [2]:
rng = np.random.default_rng(42)
X = rng.normal(size=(2000, 10))
# parte lineal (x0, x1) + parte de interacción (x2 y x3 con el mismo signo) + ruido logístico
logit = 1.5 * (1.5 * X[:, 0] - 1.5 * X[:, 1] + 2 * np.sign(X[:, 2] * X[:, 3]))
y = (rng.random(2000) < 1 / (1 + np.exp(-logit))).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X_train.shape, X_test.shape, f"proporción de la clase 1: {y.mean():.2f}")

(1200, 10) (800, 10) proporción de la clase 1: 0.49


In [3]:
base = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "kNN (k=25)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)),
    "Random Forest": RandomForestClassifier(min_samples_leaf=5, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}
filas = {}
for nombre, modelo in base.items():
    puntuaciones = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="f1_macro")
    filas[nombre] = {"F1 macro (media)": puntuaciones.mean(), "desviación": puntuaciones.std()}
pd.DataFrame(filas).T.round(3)

,F1 macro (media),desviación
Regresión logística,0.671,0.030
kNN (k=25),0.740,0.016
Random Forest,0.756,0.034
HistGradientBoosting,0.823,0.011


El *boosting* (`HistGradientBoosting`) es con diferencia el mejor modelo base ($F_1$ = 0,823), seguido del bosque (0,756) y de kNN (0,740); la regresión logística (0,671) solo captura la parte lineal y se queda lejos. Es un escenario con un modelo dominante y otros tres claramente peores: veamos qué puede aportar un metamodelo.

### 3.2. Stacking a mano y la fuga de datos

Construimos el stacking siguiendo los pasos de 2.2 con tres modelos (regresión logística, un bosque sin límite de profundidad y 1-NN, que se aprende el entrenamiento de memoria) y comparamos dos metamodelos idénticos: uno entrenado con las predicciones **dentro de muestra** (los modelos predicen los mismos datos con los que se entrenaron) y otro con las **fuera de muestra** (`cross_val_predict`).

In [4]:
modelos_fuga = {
    "lr": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "rf": RandomForestClassifier(random_state=42, n_jobs=-1),   # bosque sin límite de profundidad
    "1nn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),  # memoriza el entrenamiento
}
Z_dentro, Z_oof, Z_test = [], [], []
for modelo in modelos_fuga.values():
    Z_oof.append(cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1])
    modelo.fit(X_train, y_train)
    Z_dentro.append(modelo.predict_proba(X_train)[:, 1])   # predicciones sobre los datos con los que se entrenó
    Z_test.append(modelo.predict_proba(X_test)[:, 1])
Z_dentro, Z_oof, Z_test = (np.column_stack(z) for z in (Z_dentro, Z_oof, Z_test))

filas = {}
for nombre, Z in [("Con predicciones dentro de muestra (fuga)", Z_dentro), ("Con predicciones fuera de muestra", Z_oof)]:
    meta = LogisticRegression().fit(Z, y_train)
    filas[nombre] = {"F1 en entrenamiento": f1_score(y_train, meta.predict(Z), average="macro"),
                     "F1 en test": f1_score(y_test, meta.predict(Z_test), average="macro"),
                     **{f"peso {n}": c for n, c in zip(modelos_fuga, meta.coef_[0])}}
pd.DataFrame(filas).T.round(2)

,F1 en entrenamiento,F1 en test,peso lr,peso rf,peso 1nn
Con predicciones dentro de muestra (fuga),1.00,0.67,0.51,4.11,6.19
Con predicciones fuera de muestra,0.77,0.78,0.06,5.95,0.62


Con predicciones dentro de muestra, el metamodelo ve que el $F_1$ en entrenamiento es **1,00** (1-NN y el bosque se saben el entrenamiento) y reparte el peso entre ellos (6,19 y 4,11, frente a 0,51 de la regresión logística); en test cae a **0,67**. Con predicciones fuera de muestra el $F_1$ en entrenamiento es honesto (0,77), el peso se desplaza al bosque (5,95), 1-NN baja a 0,62 y la regresión logística casi desaparece (0,06), y en test **mejora a 0,78**. Esos 0,11 de diferencia en $F_1$ vienen solo de **con qué predicciones se entrena el metamodelo**, no del modelo ni de los datos. Los pesos están en escala de logit y las metavariables están correlacionadas, así que sirven para ver la tendencia, no para interpretarlos uno a uno.

### 3.3. `StackingClassifier`

`StackingClassifier` hace el proceso de 2.2 por nosotros: `cv=5` genera las predicciones fuera de muestra y después reentrena los modelos base con todo el entrenamiento. Usa los cuatro modelos base de 3.1 y una regresión logística como metamodelo. Comparamos en test.

In [5]:
estimadores = [(clave, modelo) for clave, modelo in zip(["lr", "knn", "rf", "hgb"], base.values())]
stacking = StackingClassifier(estimators=estimadores, final_estimator=LogisticRegression(max_iter=1000), cv=5)
stacking.fit(X_train, y_train)

filas = {nombre: f1_score(y_test, modelo.fit(X_train, y_train).predict(X_test), average="macro") for nombre, modelo in base.items()}
filas["Stacking (metamodelo: regresión logística)"] = f1_score(y_test, stacking.predict(X_test), average="macro")
print(pd.Series(filas, name="F1 macro en test").round(3))
print("\nPesos del metamodelo (probabilidad de la clase 1 de cada modelo base):")
print(pd.Series(stacking.final_estimator_.coef_[0], index=["lr", "knn", "rf", "hgb"]).round(2))

Regresión logística                           0.710
kNN (k=25)                                    0.762
Random Forest                                 0.770
HistGradientBoosting                          0.845
Stacking (metamodelo: regresión logística)    0.849
Name: F1 macro en test, dtype: float64

Pesos del metamodelo (probabilidad de la clase 1 de cada modelo base):
lr     0.09
knn    1.49
rf    -0.08
hgb    4.29
dtype: float64


El stacking obtiene 0,849 frente a 0,845 del *boosting* solo: una diferencia de 0,004, unos 3 ejemplos de 800 en test, que no permite decir que mejora. El metamodelo ha hecho lo razonable: casi todo el peso para `hgb` (4,29), algo para kNN (1,49) y prácticamente nada para la regresión logística y el bosque (el bosque es muy parecido a `hgb`, con correlación 0,81 entre sus predicciones, ver 3.4). Es decir, **aprende quién manda**; no inventa información nueva.

### 3.4. Diversidad: ¿qué ocurre según los modelos que apilas?

Primero, las correlaciones entre las probabilidades fuera de muestra de cada modelo (cuanto más bajas, más complementarios). Después, el stacking con distintos grupos de modelos, evaluado con validación cruzada sobre **todos** los datos (el `StackingClassifier` entero va dentro de `cross_val_score`, así que cada partición de test es ajena al metamodelo).

In [6]:
oof = pd.DataFrame({n: cross_val_predict(m, X_train, y_train, cv=cv, method="predict_proba")[:, 1] for n, m in base.items()})
print("Correlación entre las probabilidades fuera de muestra:")
print(oof.corr().round(2).to_string())

est = dict(zip(["lr", "knn", "rf", "hgb"], base.values()))
def apilar(claves, **kw):
    return StackingClassifier([(k, est[k]) for k in claves], final_estimator=LogisticRegression(max_iter=1000), cv=5, **kw)

filas = {}
grupos = {"Stacking: lr + knn": ["lr", "knn"], "Stacking: lr + knn + rf": ["lr", "knn", "rf"],
          "Stacking: lr + knn + rf + hgb": ["lr", "knn", "rf", "hgb"]}
for nombre, claves in grupos.items():
    puntuaciones = cross_val_score(apilar(claves), X, y, cv=cv, scoring="f1_macro")
    filas[nombre] = {"F1 macro (media)": puntuaciones.mean(), "desviación": puntuaciones.std()}
for clave in ["rf", "hgb"]:
    puntuaciones = cross_val_score(est[clave], X, y, cv=cv, scoring="f1_macro")
    filas[f"Solo {clave}"] = {"F1 macro (media)": puntuaciones.mean(), "desviación": puntuaciones.std()}
pd.DataFrame(filas).T.round(3)

Correlación entre las probabilidades fuera de muestra:
                      Regresión logística  kNN (k=25)  Random Forest  HistGradientBoosting
Regresión logística                  1.00        0.75           0.87                  0.63
kNN (k=25)                           0.75        1.00           0.82                  0.73
Random Forest                        0.87        0.82           1.00                  0.81
HistGradientBoosting                 0.63        0.73           0.81                  1.00


,F1 macro (media),desviación
Stacking: lr + knn,0.766,0.009
Stacking: lr + knn + rf,0.790,0.020
Stacking: lr + knn + rf + hgb,0.855,0.020
Solo rf,0.781,0.023
Solo hgb,0.855,0.019


La regresión logística y el *boosting* son los menos correlacionados (0,63): son los más complementarios (uno ve la parte lineal, el otro la interacción). El bosque se parece más a todos los demás (0,87 con la regresión logística, 0,81 con el *boosting*). Los resultados del apilado siguen la calidad del mejor modelo que contiene: `lr + knn` da 0,766; con el bosque, 0,790 (frente a 0,781 del bosque solo); y con el *boosting*, **0,855, igual que el *boosting* solo** (0,855). Todas las desviaciones entre particiones están en torno a 0,02, por encima de esas diferencias. Conclusión: el stacking **iguala al mejor modelo que le des** y, con suerte, lo supera un poco; su valor práctico es que **no tienes que saber de antemano cuál es el mejor**, no que lo supere con claridad.

### 3.5. Variantes: `passthrough` (cascading), metamodelo y votación

Con los cuatro modelos base probamos: el metamodelo lineal frente a un bosque, con y sin `passthrough=True` (cascading), y la votación suave (`VotingClassifier`, que promedia probabilidades con pesos iguales, sin metamodelo). Se muestra también el tiempo de la validación cruzada completa.

In [7]:
def bosque_meta():
    return RandomForestClassifier(min_samples_leaf=5, random_state=42, n_jobs=-1)

todos = [(k, est[k]) for k in est]
variantes = {
    "Stacking (metamodelo lineal)": apilar(list(est)),
    "Stacking + passthrough (metamodelo lineal)": apilar(list(est), passthrough=True),
    "Stacking (metamodelo bosque)": StackingClassifier(todos, final_estimator=bosque_meta(), cv=5),
    "Cascading: passthrough (metamodelo bosque)": StackingClassifier(todos, final_estimator=bosque_meta(), cv=5, passthrough=True),
    "Votación suave (pesos iguales)": VotingClassifier(todos, voting="soft"),
}
filas = {}
for nombre, modelo in variantes.items():
    inicio = time.perf_counter()
    puntuaciones = cross_val_score(modelo, X, y, cv=cv, scoring="f1_macro")
    filas[nombre] = {"F1 macro (media)": puntuaciones.mean(), "desviación": puntuaciones.std(),
                     "segundos": time.perf_counter() - inicio}
inicio = time.perf_counter()
puntuaciones = cross_val_score(est["hgb"], X, y, cv=cv, scoring="f1_macro")
filas["Solo hgb (referencia)"] = {"F1 macro (media)": puntuaciones.mean(), "desviación": puntuaciones.std(),
                                  "segundos": time.perf_counter() - inicio}
pd.DataFrame(filas).T.round(3)

,F1 macro (media),desviación,segundos
Stacking (metamodelo lineal),0.855,0.020,8.652
Stacking + passthrough (metamodelo lineal),0.852,0.022,8.705
Stacking (metamodelo bosque),0.861,0.013,9.394
Cascading: passthrough (metamodelo bosque),0.857,0.020,10.202
Votación suave (pesos iguales),0.829,0.010,1.618
Solo hgb (referencia),0.855,0.019,0.754


Con la regresión logística como metamodelo, `passthrough` no cambia nada (0,852 frente a 0,855), como se esperaba en 2.4: un metamodelo lineal solo puede sumar las variables originales, no condicionar a ellas. Con un bosque como metamodelo el resultado es 0,861 sin `passthrough` y 0,857 con él (cascading); las diferencias entre las variantes (≤ 0,009) son menores que la desviación entre particiones (0,013–0,022), así que **no hay evidencia de que el cascading mejore** en este conjunto. La votación suave (0,829) queda por debajo: da el mismo peso a los modelos débiles y al bueno, mientras que el metamodelo aprende a ignorarlos.

El coste es lo más visible: el stacking tarda unos 9 s en la validación cruzada completa, frente a 0,6 s del *boosting* solo y 1,4 s de la votación (los tiempos dependen del equipo, pero la proporción, un orden de magnitud, se mantiene).

### 3.6. Un conjunto real y limpio: breast cancer

Mismos cuatro modelos y mismo metamodelo, validación cruzada de 5 particiones.

In [8]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
base_bc = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "kNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}
resultados = {n: cross_val_score(m, X_bc, y_bc, cv=cv, scoring="f1_macro") for n, m in base_bc.items()}
stacking_bc = StackingClassifier(list(base_bc.items()), final_estimator=LogisticRegression(max_iter=1000), cv=5)
resultados["Stacking"] = cross_val_score(stacking_bc, X_bc, y_bc, cv=cv, scoring="f1_macro")
pd.DataFrame({"F1 macro (media)": {n: r.mean() for n, r in resultados.items()},
              "desviación": {n: r.std() for n, r in resultados.items()}}).round(3)

,F1 macro (media),desviación
Regresión logística,0.971,0.018
kNN,0.960,0.020
Random Forest,0.953,0.013
HistGradientBoosting,0.964,0.014
Stacking,0.977,0.018


Los cinco resultados están entre 0,953 y 0,977, con desviaciones de 0,013–0,020: el stacking (0,977) está en la parte alta, pero la diferencia con la regresión logística (0,971) es de 0,006, bien dentro de la desviación. En un conjunto pequeño y casi linealmente separable, **no hay nada que combinar**: cualquier modelo razonable ya lo resuelve.

## 4. Parámetros importantes y errores comunes

### 4.1. Parámetros de `StackingClassifier`

| Parámetro | Qué controla | Consejo |
|---|---|---|
| `estimators` | Lista de pares `(nombre, modelo)` base | Modelos de familias distintas y razonablemente buenos; el preprocesamiento de cada uno, dentro de su `Pipeline` |
| `final_estimator` | Metamodelo | Por defecto, regresión logística. Empieza por uno simple; un bosque solo si usas `passthrough` |
| `cv` | Cómo se generan las predicciones fuera de muestra | 5 es razonable (un entero o un objeto de validación cruzada); nunca lo sustituyas por predicciones sobre los mismos datos de entrenamiento, que es la fuga de 3.2 |
| `stack_method` | Qué salida de cada modelo base se usa | `"auto"` elige `predict_proba`, luego `decision_function`, luego `predict`; con probabilidades el metamodelo recibe la confianza |
| `passthrough` | Añade las variables originales al metamodelo | `True` = cascading; requiere un metamodelo capaz de usarlas (árbol, bosque, *boosting*) |
| `n_jobs` | Paralelismo en el ajuste de los modelos base | Útil, porque cada modelo se entrena $K+1$ veces |
| `StackingRegressor` | Versión para regresión | Mismos parámetros; metamodelo típico: `RidgeCV` |

### 4.2. Errores comunes

**1. Entrenar el metamodelo con predicciones dentro de muestra.** Es el error principal y lo hemos visto en 3.2: con predicciones de los propios datos de entrenamiento, el metamodelo pone el peso en los modelos que memorizan (1-NN y bosque) y empeora en test (0,67 frente a 0,78) aunque en entrenamiento parezca perfecto ($F_1$ = 1,00). Usa siempre `cross_val_predict` o `StackingClassifier(cv=...)`.

**2. Medir el stacking con el mismo test que se usó para elegir.** Si ajustas modelos base, metamodelo y grupo de modelos mirando el test, ya no es independiente. Evalúa el `StackingClassifier` completo con validación cruzada (como en 3.4) o con un test que se mire una sola vez, como en el [flujo de ML](01-flujo-ml-y-evaluacion.ipynb).

**3. Apilar modelos débiles o casi iguales y esperar milagros.** En un proyecto propio con regresión logística, un árbol, kNN y *Naive Bayes* como base, el stacking quedó **por debajo** de un Random Forest y de un Gradient Boosting entrenados solos. Es lo que se ve en 3.4: sin un modelo fuerte entre los base, el metamodelo no puede superar al mejor de los que ve; con `lr + knn` el $F_1$ se queda en 0,766 y con el *boosting* dentro llega a 0,855. Antes de apilar, mira la correlación entre las predicciones de los modelos (3.4).

**4. Un metamodelo demasiado complejo.** Con 4 o 5 metavariables muy correlacionadas, un bosque o una red como metamodelo se sobreajusta con facilidad. Empieza con una regresión logística y compáralo con la votación simple.

**5. Preprocesar fuera del `Pipeline` de cada modelo.** Los modelos lineales y los vecinos necesitan escalado; los árboles no. Si estandarizas todo antes de apilar, haces más trabajo del necesario y puedes meter fuga si ajustas el escalador con todos los datos. Pon el preprocesamiento de cada modelo dentro de su propio `Pipeline`, como en 3.1.

**6. Ignorar el coste.** El stacking entrena cada modelo $K+1$ veces y luego predice con todos. En 3.5 el tiempo es un orden de magnitud mayor que el de un único *boosting* (≈ 9 s frente a 0,6 s): la mejora tiene que compensar ese coste en entrenamiento, en predicción y en mantenimiento.

**7. Leer los pesos del metamodelo como importancia de variables.** Son pesos sobre **modelos**, no sobre variables, y con metavariables correlacionadas (3.4) no se pueden interpretar uno a uno; para saber qué variables usa el conjunto hace falta la importancia por permutación sobre el modelo completo ([interpretabilidad](14-interpretabilidad.ipynb)).

**8. Evaluar solo con la exactitud cuando las clases están desbalanceadas.** Igual que en [Random Forest](10-bagging-y-random-forest.ipynb) y [Boosting](11-boosting.ipynb): usa $F_1$ macro o una métrica por clase, y ver las técnicas de [muestreo y desbalanceo](../03-preparacion-datos/04-muestreo-y-desbalanceo.ipynb).

## 5. Comparación con métodos relacionados

| Método | Cómo combina | Modelos base | Entrenamiento | Cuándo usarlo |
|---|---|---|---|---|
| [Random Forest](10-bagging-y-random-forest.ipynb) | Promedio en paralelo | Árboles profundos | Paralelo, barato | Primera opción robusta con poco ajuste |
| [Boosting](11-boosting.ipynb) | Suma secuencial de árboles débiles | Árboles pequeños | Secuencial | Máxima precisión en datos tabulares, con ajuste cuidadoso |
| Votación (`VotingClassifier`) | Promedio con pesos fijos | Cualquiera | Un entrenamiento por modelo | Modelos de calidad parecida; sin riesgo de fuga ni metamodelo que ajustar |
| **Stacking** | Un metamodelo aprende los pesos | Cualquiera, de familias distintas | $K+1$ entrenamientos por modelo | Cuando tienes modelos buenos y complementarios y el último punto de rendimiento importa |
| **Cascading** (`passthrough=True`) | Como el stacking, con las variables originales | Cualquiera | Igual que el stacking | Cuando la fiabilidad de cada modelo depende de la zona del espacio; con metamodelo no lineal |

**Cuándo elegir stacking.** Cuando ya has llevado cada modelo a un buen nivel, ninguno domina claramente y el coste extra es asumible (competiciones, sistemas donde un punto de $F_1$ vale mucho). Si un único modelo, como un *boosting* bien ajustado, ya es claramente el mejor, el stacking aporta poco más que robustez; si el modelo tiene que ser explicable o rápido, elige un único modelo.

## 6. Referencias

### Fuentes

- scikit-learn: [*Ensembles: Gradient boosting, random forests, bagging, voting, stacking*](https://scikit-learn.org/stable/modules/ensemble.html) (apartados de votación y *stacking*, `passthrough`, `stack_method`).
- scikit-learn: [`StackingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html), [`StackingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html), [`VotingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) y [`HistGradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html).
- scikit-learn: [`cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html) y [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html).
- scikit-learn: [`load_breast_cancer`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).
- UCI Machine Learning Repository: [*Breast Cancer Wisconsin (Diagnostic)*](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), origen del conjunto de datos.

### Para saber más

- Wolpert, D. H. (1992). *Stacked generalization*. Neural Networks, 5(2), 241–259. [doi:10.1016/S0893-6080(05)80023-1](https://doi.org/10.1016/S0893-6080(05)80023-1). El artículo que introduce el stacking, con la idea de entrenar el segundo nivel con predicciones obtenidas sin usar el propio ejemplo.
- Breiman, L. (1996). *Stacked regressions*. Machine Learning, 24(1), 49–64. [doi:10.1007/BF00117832](https://doi.org/10.1007/BF00117832). Estudia el stacking en regresión y muestra por qué conviene un metamodelo simple y con pesos restringidos.